# 01 - Exploration des datasets Darija

Ce notebook explore les deux datasets utilises dans le projet :

1. **`atlasia/DODa-audio-dataset`** -> ASR (audio + transcription Latin/Arabe + traduction EN)
2. **`MBZUAI-Paris/Darija-SFT-Mixture`** -> Traduction (on filtre le sous-ensemble darija -> francais)

**A executer sur Google Colab** (Runtime > Modifier le type d'execution > GPU, pas obligatoire pour cette phase
mais utile pour le notebook suivant) ou sur ta machine perso avec une connexion internet.


## 1. Installation des dependances

In [ ]:
!pip install -q datasets soundfile librosa pandas matplotlib

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
import numpy as np

pd.set_option("display.max_colwidth", 120)

## 1bis. Connexion a Google Drive (stockage persistant)

On monte ton Google Drive pour que toutes les donnees telechargees et tous les fichiers
generes (CSV filtres, futurs checkpoints de modeles) soient sauvegardes de facon persistante,
meme si la session Colab se deconnecte.

A l'execution, une fenetre d'autorisation Google va s'ouvrir : autorise l'acces.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Dossier racine du projet sur ton Drive (modifiable si tu preferes un autre nom/emplacement)
PROJECT_DIR = "/content/drive/MyDrive/darija-transcription-translation"

# Creation de l'arborescence si elle n'existe pas encore
for sub in ["data/raw", "data/processed", "models"]:
    os.makedirs(os.path.join(PROJECT_DIR, sub), exist_ok=True)

print("Projet sur Drive pret a :", PROJECT_DIR)
print(os.listdir(PROJECT_DIR))

## 1ter. Authentification HuggingFace (necessaire pour dataset gated)

Le dataset `atlasia/DODa-audio-dataset` est en acces "gated" sur le Hub (protection liee aux donnees vocales).
Etapes a faire UNE SEULE FOIS avant d'executer la cellule ci-dessous :

1. Creer un compte sur https://huggingface.co/join (si pas deja fait)
2. Aller sur https://huggingface.co/datasets/atlasia/DODa-audio-dataset et accepter les conditions d'acces
3. Creer un token (type "Read") sur https://huggingface.co/settings/tokens
4. Dans Colab : cliquer sur l'icone cle (Secrets) a gauche -> ajouter un secret nomme `HF_TOKEN` avec ton token comme valeur, activer "Notebook access"
5. Executer la cellule ci-dessous


In [ ]:
from huggingface_hub import login

try:
    # Sur Colab : recupere le token depuis les Secrets
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except (ImportError, Exception):
    # En local (pas Colab) : decommenter et coller le token directement, ou utiliser une variable d'env
    import os
    hf_token = os.environ.get('HF_TOKEN')

if hf_token:
    login(token=hf_token)
    print("Authentification HuggingFace reussie.")
else:
    print("ATTENTION: HF_TOKEN non trouve. Voir les instructions ci-dessus.")

## 2. Dataset ASR : `atlasia/DODa-audio-dataset`

Contient 12 743 echantillons audio + texte darija (script Latin et Arabe) + traduction anglaise.

In [ ]:
import os
from datasets import load_dataset, load_from_disk

asr_cache_path = os.path.join(PROJECT_DIR, "data/raw/doda_audio_dataset")

if os.path.exists(asr_cache_path):
    print("Chargement depuis le cache Drive (pas de re-telechargement)...")
    asr_ds = load_from_disk(asr_cache_path)
else:
    print("Telechargement depuis le Hub HuggingFace...")
    asr_ds = load_dataset("atlasia/DODa-audio-dataset")
    print("Sauvegarde sur Drive pour la prochaine fois...")
    asr_ds.save_to_disk(asr_cache_path)

print(asr_ds)

In [ ]:
# Inspection des colonnes disponibles et du premier exemple
split_name = list(asr_ds.keys())[0]
print("Colonnes:", asr_ds[split_name].column_names)
print()

example = asr_ds[split_name][0]


def get_audio_array_and_sr(audio_value, debug=False):
    """Gere plusieurs formats possibles selon la version de `datasets` :
    - dict classique {"array": ..., "sampling_rate": ...} (anciennes versions)
    - objet AudioDecoder (datasets >= 5.x, backend torchcodec)
    """
    if isinstance(audio_value, dict):
        arr, sr = audio_value["array"], audio_value["sampling_rate"]
        if debug:
            print(f"[debug] format dict -> arr.shape={getattr(arr, 'shape', len(arr))}, sr={sr}")
        return arr, sr

    if debug:
        print(f"[debug] type audio_value: {type(audio_value)}")
        print(f"[debug] attributs disponibles: {[a for a in dir(audio_value) if not a.startswith('_')]}")

    # Cas AudioDecoder (torchcodec) : on recupere tous les samples
    samples = audio_value.get_all_samples()

    if debug:
        print(f"[debug] type samples: {type(samples)}")
        print(f"[debug] attributs samples: {[a for a in dir(samples) if not a.startswith('_')]}")
        print(f"[debug] samples.data.shape (brut): {samples.data.shape}")
        print(f"[debug] samples.sample_rate: {samples.sample_rate}")

    array = samples.data.numpy()
    # samples.data a la forme (channels, num_samples) -> on prend la moyenne des canaux si stereo,
    # ou on aplatit si mono, plutot qu'un simple squeeze() qui peut mal se comporter
    if array.ndim == 2:
        array = array.mean(axis=0)  # downmix mono si plusieurs canaux
    sr = samples.sample_rate

    if debug:
        print(f"[debug] array.shape (final): {array.shape}, sr final: {sr}")

    return array, sr


# Premier appel en mode debug pour bien voir ce qui se passe
print("=== DIAGNOSTIC sur le premier exemple ===")
arr_debug, sr_debug = get_audio_array_and_sr(example["audio"], debug=True)
print(f"=== Resultat: shape={arr_debug.shape}, sr={sr_debug}, duree={len(arr_debug)/sr_debug:.3f}s ===")
print()

for k, v in example.items():
    if k != "audio":
        print(f"{k}: {v}")

In [ ]:
# Ecouter un exemple audio (fonctionne dans Colab/Jupyter)
import IPython.display as ipd

audio_array, sr = get_audio_array_and_sr(example["audio"])

print("Texte (Latin)        :", example.get("darija_Latn", "N/A"))
print("Texte (Arabe, new)   :", example.get("darija_Arab_new", "N/A"))
print("Texte (Arabe, old)   :", example.get("darija_Arab_old", "N/A"))
print("Traduction EN        :", example.get("english", "N/A"))

ipd.Audio(audio_array, rate=sr)

In [ ]:
# Statistiques generales : duree audio totale, distribution des longueurs de texte
n_sample = min(500, len(asr_ds[split_name]))  # echantillon pour aller vite

durations = []
for i in range(n_sample):
    arr, sr_i = get_audio_array_and_sr(asr_ds[split_name][i]["audio"])
    durations.append(len(arr) / sr_i)

print("Nombre total d'exemples:", len(asr_ds[split_name]))
print(f"Duree min: {min(durations):.3f}s, Duree max: {max(durations):.3f}s")
print(f"Duree moyenne (sur {n_sample} ech.): {np.mean(durations):.2f}s")
print(f"Duree totale estimee (extrapolee): {np.mean(durations) * len(asr_ds[split_name]) / 3600:.2f}h")

if np.mean(durations) == 0:
    print()
    print("ATTENTION: duree moyenne = 0, quelque chose ne va pas dans l'extraction.")
    print("Relancer la cellule precedente avec debug=True pour investiguer:")
    print('  get_audio_array_and_sr(asr_ds[split_name][0]["audio"], debug=True)')

plt.figure(figsize=(8,4))
plt.hist(durations, bins=30)
plt.title("Distribution des durees audio (echantillon)")
plt.xlabel("Duree (secondes)")
plt.ylabel("Nombre d'exemples")
plt.show()

## 3. Dataset traduction : `MBZUAI-Paris/Darija-SFT-Mixture`

Contient 50 760 instructions de traduction sur 6 directions (darija<->EN, FR, MSA).
On va filtrer uniquement les paires **darija -> francais**.

In [ ]:
translation_cache_path = os.path.join(PROJECT_DIR, "data/raw/darija_sft_mixture_full")

if os.path.exists(translation_cache_path):
    print("Chargement depuis le cache Drive (pas de re-telechargement)...")
    translation_ds = load_from_disk(translation_cache_path)
else:
    print("Telechargement depuis le Hub HuggingFace...")
    translation_ds = load_dataset("MBZUAI-Paris/Darija-SFT-Mixture")
    print("Sauvegarde sur Drive pour la prochaine fois...")
    translation_ds.save_to_disk(translation_cache_path)

print(translation_ds)

In [ ]:
split_name_t = list(translation_ds.keys())[0]
print("Colonnes:", translation_ds[split_name_t].column_names)
print()
print(translation_ds[split_name_t][0])
print()

# Conversion en DataFrame pandas (necessaire pour les cellules suivantes : value_counts, filtre, etc.)
df_t = translation_ds[split_name_t].to_pandas()
print(f"df_t cree avec succes : {len(df_t)} lignes")

In [ ]:
# La colonne "direction" indique le sens de traduction (ex: dr_en, en_dr, dr_fr, fr_dr...)
# On regarde toutes les valeurs uniques pour identifier le(s) bon(s) code(s) pour le francais
print("Valeurs uniques de la colonne direction:")
print(df_t["direction"].value_counts())

In [ ]:
# Filtre strict sur la colonne "direction" (pas de recherche heuristique dans le texte,
# qui generait trop de faux positifs - cf. "l'institut francais" dans un resume MArSum)
#
# On garde uniquement dr_fr (darija -> francais), le sens qui nous interesse pour ce projet
# (transcription audio darija, puis traduction VERS le francais).
# A AJUSTER si la cellule precedente montre un autre nom de code pour cette direction.
FRENCH_DIRECTION = "dr_fr"  # <-- ajuster selon la sortie de la cellule precedente si besoin

df_fr = df_t[df_t["direction"] == FRENCH_DIRECTION].copy()
print(f"Nombre de paires darija -> francais (direction == '{FRENCH_DIRECTION}'): {len(df_fr)}")


def extract_user_assistant(messages):
    """Extrait le texte 'user' (instruction + phrase source) et 'assistant' (traduction)
    depuis la liste de dicts {'role': ..., 'content': ...} de la colonne messages."""
    user_text, assistant_text = None, None
    for m in messages:
        if m.get("role") == "user":
            user_text = m.get("content")
        elif m.get("role") == "assistant":
            assistant_text = m.get("content")
    return user_text, assistant_text


df_fr[["source_text", "target_text"]] = df_fr["messages"].apply(
    lambda msgs: pd.Series(extract_user_assistant(msgs))
)

df_fr[["direction", "source_text", "target_text"]].head(10)

In [ ]:
# Sauvegarde du sous-ensemble filtre et nettoye pour la phase de preprocessing
# On ne garde que les colonnes utiles (pas "messages" brut, qui contient des objets imbriques)
import os

output_path = os.path.join(PROJECT_DIR, "data/processed/translation_pairs_fr_raw.csv")

cols_to_keep = ["dataset", "id", "direction", "source_text", "target_text"]
df_fr[cols_to_keep].to_csv(output_path, index=False)

print(f"Sauvegarde -> {output_path}")
print(f"Nombre de paires sauvegardees: {len(df_fr)}")

## 4. Prochaines etapes

- Valider manuellement un echantillon des paires darija->francais extraites (qualite, bruit)
- Passer a `scripts/preprocess_asr_data.py` et `scripts/preprocess_translation_data.py` (Phase 2)
- Decider : un seul modele Whisper multi-script (Latin+Arabe) ou deux modeles separes ?
